# Udemy Courses Dataset: Preprocessing

## 1. Notebook Goal

This notebook prepares a clean recommendation-ready version of the Udemy Courses dataset. The processed dataset will be used later by the TF-IDF and KNN recommenders, so the goal is to keep the transformation simple, transparent, and reproducible.


## 2. Import Libraries

Only standard data preparation libraries are needed at this stage. Text cleaning is intentionally basic because advanced NLP steps such as stemming or lemmatization are out of scope for the first preprocessing version.


In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd


## 3. Load Raw Dataset

The raw dataset is loaded from `data/raw/udemy_courses.csv`. A fallback path is included so the notebook can still run if it is opened from inside the `notebooks` folder.


In [ ]:
raw_data_path = Path("data/raw/udemy_courses.csv")
if not raw_data_path.exists():
    raw_data_path = Path("../data/raw/udemy_courses.csv")

df_raw = pd.read_csv(raw_data_path)
df_raw.head()


## 4. Select Recommendation Features

The recommender models do not need every column from the raw dataset. We keep the main course identifier, text/category fields, and numeric popularity or course-size indicators. Columns such as URL and publication timestamp are excluded because they are not required for the first recommender versions.


In [ ]:
recommendation_columns = [
    "course_id",
    "course_title",
    "subject",
    "level",
    "num_subscribers",
    "num_reviews",
    "price",
    "content_duration",
]

df = df_raw[recommendation_columns].copy()
df.head()


## 5. Initial Data Quality Snapshot

Before changing the data, we check missing values and duplicates. This gives us a clear before/after comparison for the preprocessing steps.


In [ ]:
initial_quality = pd.DataFrame({
    "missing_values": df.isna().sum(),
    "missing_share": (df.isna().mean() * 100).round(2),
    "dtype": df.dtypes.astype(str),
})

print(f"Initial rows: {len(df):,}")
print(f"Exact duplicate rows: {df.duplicated().sum():,}")
print(f"Duplicate course IDs: {df.duplicated(subset='course_id').sum():,}")
initial_quality


## 6. Basic Text Cleaning

The text cleaning is intentionally simple and explainable: lowercase text, remove punctuation that can create noisy tokens, and normalize repeated spaces. This is enough for the first content-based recommender input without introducing advanced NLP complexity.


In [ ]:
def clean_text(value):
    if pd.isna(value):
        return np.nan
    value = str(value).lower()
    value = re.sub(r"[^a-z0-9+.#\s]", " ", value)
    value = re.sub(r"\s+", " ", value).strip()
    return value

text_columns = ["course_title", "subject", "level"]
for column in text_columns:
    df[column] = df[column].apply(clean_text)

df[text_columns].head()
